# LARES — ACSAC 2026 Artifact Evaluation (Google Colab)

**Paper:** *LARES: Host-centered Lateral Movement Detection via Inductive Graph Reasoning*

One linear pipeline: paste the dataset link in section 1, then **Runtime → Run all**. The
notebook clones the code and released weights, downloads the preprocessed dataset from
Google Drive, compiles the graph snapshots, evaluates from the weights, and prints the
reproduced numbers next to the published ones.

| | |
|---|---|
| Disk | ~35 GB used, out of the ~100 GB a Colab runtime provides |
| Download | 14.35 GB (the preprocessed LANL and OpTC dataset) |
| Runtime | several hours end to end; snapshot compilation dominates |
| Hardware | a CPU runtime works; a T4 GPU speeds up the evaluation |

Restricting `DATASETS` to `['LANL']` in section 1 gives the shortest complete run.

## Claims reproduced

1. **Precision under class imbalance (§V-B, Figure 2, Table II).** 70% edge-level precision
   on LANL with 104 false positives, against 1,408 for ARGUS and 9,286 for EULER.
2. **Inductive detection (§V-C, Table II, Exp1-Exp3).** Detection holds when 30% and 50% of
   hosts, including all malicious ones, are unseen during training.
3. **Source host localization (§E, Table VII).** One false positive among 13.2k LANL hosts,
   and 2 of 3 root attack hosts on OpTC with a single false positive.
4. **Two-stage detection (§V-E, Table III, *Detection* row).** Replacing it with direct edge
   thresholding collapses precision.

---
## 1. Configuration

`DATASET_URL` is a Google Drive share link to `lanl_optc_datasets.tar.gz` (or `.zip`),
shared as *Anyone with the link*. Everything else can stay as it is.

In [ ]:
import os

DATASET_URL  = ''    # e.g. 'https://drive.google.com/file/d/<id>/view?usp=sharing'

REPO_GIT_URL = 'https://github.com/TristanBilot/lares.git'
REPO_DIR     = '/content/lares'

DATASETS     = ['LANL', 'OPTC']   # ['LANL'] for the shortest complete run
EXPERIMENTS  = [0, 1, 2, 3]       # Exp0 transductive, Exp1-Exp3 inductive

DATA_ROOT = '/content/lanl_optc_datasets'
os.environ['LARES_DATA_ROOT'] = DATA_ROOT
print('Data root:', DATA_ROOT)

---
## 2. Code and dependencies

Clones the repository, whose `weights/` folder holds the released model weights, and adds
the two packages Colab does not ship. The compiled PyG extensions (`torch_scatter`,
`torch_sparse`, ...) are not needed: nothing in `src/` imports them.

In [ ]:
import subprocess

def sh(cmd, check=True):
    """Run a shell command, streaming its output into the notebook.

    subprocess writes to the kernel's file descriptors, which Colab does not show in the
    cell, so output is piped back and printed here.
    """
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'command failed with exit code {code}: {cmd}')
    return code

sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', check=False)

if not os.path.isfile(os.path.join(REPO_DIR, 'src', 'main.py')):
    sh(f'git clone --depth 1 {REPO_GIT_URL} {REPO_DIR}')
os.chdir(REPO_DIR)

sh('pip -q install torch_geometric==2.8.0.post1 wandb==0.30.0', check=False)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

# Fail here, with a readable message, rather than inside a subprocess later.
sh('python -c "'
   'import torch, torch_geometric, sklearn, pandas, joblib, tqdm, wandb; '
   'from libauc.losses import APLoss; '
   'import sys; sys.path.insert(0, \'src\'); '
   'from models.model import Model; '
   'print(\'all imports OK\')"')

missing = [f for d in DATASETS for e in EXPERIMENTS
           if not os.path.isfile(f := f'weights/weights_{d}_inductive_exp{e}.pkl')]
assert not missing, f'weight files missing from the repository: {missing}'
print('All weights required by DATASETS/EXPERIMENTS are present.')

---
## 3. Download and extract the dataset

Skipped automatically when the dataset is already unpacked, so an interrupted session can
be resumed by running all cells again. If `gdown` reports a quota or permission error,
check that the file is shared as *Anyone with the link*; Google also throttles files that
were downloaded by many people in the past 24 hours.

In [ ]:
import glob

def locate_data_root():
    for cand in [DATA_ROOT] + glob.glob('/content/*') + glob.glob('/content/*/*'):
        if any(os.path.isdir(os.path.join(cand, d, 'preprocessed')) for d in ['LANL', 'OPTC']):
            return cand
    return None

root = locate_data_root()
if root is None:
    assert DATASET_URL, 'Set DATASET_URL in section 1 to the Google Drive link of the dataset.'
    sh('pip -q install -U gdown', check=False)
    import gdown
    os.makedirs('/content/downloads', exist_ok=True)
    archive = gdown.download(url=DATASET_URL, output='/content/downloads/', fuzzy=True)
    assert archive, ('gdown could not download the file: check the sharing setting '
                     '("Anyone with the link") and the 24h download quota.')
    print('Downloaded:', archive)
    if archive.endswith(('.tar.gz', '.tgz', '.tar')):
        sh(f'tar -xf "{archive}" -C /content')
    elif archive.endswith('.zip'):
        sh(f'unzip -q -o "{archive}" -d /content')
    else:
        raise ValueError(f'unexpected archive type: {archive}')
    root = locate_data_root()
    assert root, 'the archive did not contain LANL/preprocessed or OPTC/preprocessed'

DATA_ROOT = root
os.environ['LARES_DATA_ROOT'] = root
print('LARES_DATA_ROOT =', root)
sh(f'du -sh {root}/*', check=False)

---
## 4. Compile the graph snapshots

The *Compile datasets* step of the README: the 1-minute CSV files become 60-minute (LANL)
and 30-minute (OpTC) graph snapshots stored as tensors, one run of `src/datasets.py` per
experiment. This is the slow part. Experiments whose snapshots already exist are skipped.

In [ ]:
def snapshots_of(dataset, exp):
    return glob.glob(os.path.join(DATA_ROOT, dataset, 'compiled', 'test',
                                  f'{dataset}_exp{exp}', '*.pkl'))

def missing_experiments():
    return [(ds, e) for ds in DATASETS for e in EXPERIMENTS if not snapshots_of(ds, e)]

for ds, e in missing_experiments():
    sh(f'python src/datasets.py --dataset={ds} --dataset_name={ds}_exp{e} '
       f'--inductive_experiment=Exp{e}')

still = missing_experiments()
assert not still, f'compilation did not produce snapshots for {still}'
for ds in DATASETS:
    print(f'{ds}: ' + ', '.join(f'Exp{e}={len(snapshots_of(ds, e))} snapshots'
                                for e in EXPERIMENTS))

---
## 5. Evaluate from the released weights

The commands of the README, section *Reproduce experiments → From weights*:

```shell
python src/main.py --config=LANL_inductive_exp0 --use_weights=True
...
python src/main.py --config=OPTC_inductive_exp3 --use_weights=True
```

Each run prints a node-level block (stage 1 of the detection, Table VII) followed by an
edge-level block (stage 2, Table II).

In [ ]:
import re, subprocess, sys, time

LOG_DIR = '/content/run_logs'
os.makedirs(LOG_DIR, exist_ok=True)

def run_config(config, extra_args='', tag=None):
    """Run one evaluation and return (stdout, wall_clock_seconds)."""
    tag = tag or config
    cmd = f'python src/main.py --config={config} --use_weights=True {extra_args}'.strip()
    print('$', cmd, flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=REPO_DIR, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.time() - start
    with open(os.path.join(LOG_DIR, f'{tag}.log'), 'w') as f:
        f.write(proc.stdout)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        raise RuntimeError(f'{cmd} failed with code {proc.returncode}')
    print(f'  done in {elapsed:.0f}s -> {LOG_DIR}/{tag}.log', flush=True)
    return proc.stdout, elapsed

NUM = r'(-?\d+\.?\d*|nan)'

def parse_block(block):
    """Extract the metrics printed by src/utils/eval.py for one detection level."""
    m = re.search(rf'AP: {NUM}.*?recall: {NUM} \| precision: {NUM} \| MCC: {NUM}', block, re.S)
    c = re.search(rf'TP: {NUM}/(\d+) \| FP: {NUM}/(\d+)', block)
    if not (m and c):
        return None
    tp, pos, fp, neg = float(c.group(1)), int(c.group(2)), float(c.group(3)), int(c.group(4))
    return {'TP': int(tp), 'FP': int(fp), 'FN': pos - int(tp), 'TN': neg - int(fp),
            'Recall': float(m.group(2)), 'Precision': float(m.group(3)),
            'MCC': float(m.group(4)), 'AP': float(m.group(1))}

def parse_output(stdout):
    """Split the run output into its node-level and edge-level metric blocks."""
    out = {}
    for key, header in [('node', 'Node detection metrics:'),
                        ('edge', 'Edge detection metrics:'),
                        ('edge', 'Standard edge-level detection:')]:
        if header in stdout:
            block = stdout.split(header)[-1]
            parsed = parse_block(block[:block.find('\n\n\n') if '\n\n\n' in block else 600])
            if parsed:
                out[key] = parsed
    return out

print('runner ready')

In [ ]:
results = {}

for ds in DATASETS:
    for e in EXPERIMENTS:
        config = f'{ds}_inductive_exp{e}'
        stdout, elapsed = run_config(config)
        parsed = parse_output(stdout)
        parsed['elapsed_s'] = elapsed
        results[(ds, f'Exp{e}')] = parsed

print('\nAll runs finished.')

---
## 5.1 Comparison with the paper

Reference values are transcribed from Table II (edge level) and Table VII (node level).
Small deviations are expected across library and hardware versions; the claims are about
orders of magnitude in false positives, so differences of a few units are not meaningful.

In [ ]:
import pandas as pd

# Table II, LARES rows (paper).
PAPER_EDGE = {
    ('LANL', 'Exp0'): dict(TP=247, FP=104, Recall=0.61, Precision=0.70, MCC=0.65, AP=0.70),
    ('LANL', 'Exp1'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63, AP=0.66),
    ('LANL', 'Exp2'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63, AP=0.66),
    ('LANL', 'Exp3'): dict(TP=241, FP=156, Recall=0.59, Precision=0.61, MCC=0.60, AP=0.62),
    ('OPTC', 'Exp0'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37, AP=0.39),
    ('OPTC', 'Exp1'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37, AP=0.39),
    ('OPTC', 'Exp2'): dict(TP=24,  FP=66,  Recall=0.41, Precision=0.27, MCC=0.33, AP=0.37),
    ('OPTC', 'Exp3'): dict(TP=27,  FP=63,  Recall=0.49, Precision=0.30, MCC=0.37, AP=0.36),
}

# Table VII, LARES rows (paper); identical across Exp0-Exp3.
PAPER_NODE = {
    'LANL': dict(TP=1, FP=1, Recall=1.00, Precision=0.50, MCC=0.71),
    'OPTC': dict(TP=2, FP=1, Recall=0.67, Precision=0.67, MCC=0.67),
}

rows = []
for (ds, exp), res in results.items():
    got, ref = res.get('edge', {}), PAPER_EDGE[(ds, exp)]
    rows.append({
        'Dataset': ds, 'Exp': exp,
        'TP': got.get('TP'), 'TP (paper)': ref['TP'],
        'FP': got.get('FP'), 'FP (paper)': ref['FP'],
        'Precision': got.get('Precision'), 'Precision (paper)': ref['Precision'],
        'Recall': got.get('Recall'), 'Recall (paper)': ref['Recall'],
        'MCC': got.get('MCC'), 'MCC (paper)': ref['MCC'],
        'Runtime (s)': round(res.get('elapsed_s', float('nan'))),
    })
edge_df = pd.DataFrame(rows)

rows = []
for (ds, exp), res in results.items():
    got, ref = res.get('node', {}), PAPER_NODE[ds]
    rows.append({
        'Dataset': ds, 'Exp': exp,
        'TP': got.get('TP'), 'TP (paper)': ref['TP'],
        'FP': got.get('FP'), 'FP (paper)': ref['FP'],
        'Precision': got.get('Precision'), 'Precision (paper)': ref['Precision'],
        'MCC': got.get('MCC'), 'MCC (paper)': ref['MCC'],
    })
node_df = pd.DataFrame(rows)

print('Edge-level detection - paper Table II (LARES rows)')
display(edge_df)
print('\nNode-level source host detection - paper Table VII (LARES rows)')
display(node_df)

edge_df.to_csv('/content/lares_table2_reproduction.csv', index=False)
node_df.to_csv('/content/lares_table7_reproduction.csv', index=False)
print('\nSaved to /content/lares_table2_reproduction.csv and /content/lares_table7_reproduction.csv')

---
## 5.2 False positives against the baselines (Figure 2)

The baselines are separate code bases and are not rerun here. Their published counts on
LANL in the transductive setting (Exp0) are the comparison point, as in Figure 2.

In [ ]:
baseline_fp = {'EULER': (297, 9286), 'ARGUS': (269, 1408), 'JBEIL': (89, 31067)}

lares = results.get(('LANL', 'Exp0'), {}).get('edge')
if lares is None:
    print('Run LANL Exp0 first (section 7).')
else:
    rows = [{'System': k, 'TP': tp, 'FP': fp, 'FP per TP': round(fp / tp, 1)}
            for k, (tp, fp) in baseline_fp.items()]
    rows.append({'System': 'LARES (this run)', 'TP': lares['TP'], 'FP': lares['FP'],
                 'FP per TP': round(lares['FP'] / max(lares['TP'], 1), 1)})
    df = pd.DataFrame(rows)
    df['FP reduction vs LARES'] = (df['FP'] / lares['FP']).round(1).astype(str) + 'x'
    display(df)

---
## 5.3 Two-stage detection ablation (Table III, *Detection* row)

`--use_direct_edge_detection=True` replaces the two-stage procedure with the direct edge
thresholding used by EULER and ARGUS, at a fixed recall. No retraining involved.

In [ ]:
ablation = []
for ds in DATASETS:
    config = f'{ds}_inductive_exp0'
    stdout, _ = run_config(config, '--use_direct_edge_detection=True',
                           tag=f'{config}_direct_edge')
    direct = parse_output(stdout).get('edge', {})
    two_stage = results[(ds, 'Exp0')]['edge']
    ablation += [
        {'Dataset': ds, 'Detection': 'two-stage (LARES)', **{k: two_stage[k]
         for k in ['TP', 'FP', 'Precision', 'Recall', 'MCC']}},
        {'Dataset': ds, 'Detection': 'direct edge thresholding', **{k: direct.get(k)
         for k in ['TP', 'FP', 'Precision', 'Recall', 'MCC']}},
    ]
display(pd.DataFrame(ablation))

---
## 6. Notes

| Paper item | Reproduced by |
|---|---|
| Table II, LARES rows | section 5.1 |
| Figure 2, FP counts on LANL | section 5.2 |
| Table VII, LARES rows | section 5.1 |
| Table III, *Detection* row | section 5.3 |
| Table III, other rows; Figures 1, 4, 6, 7, 8 | retraining: the README sections *Ablation study*, *MCC @ 10-100% of unseen hosts* and *Hyperparameter changes* run here as well, e.g. `sh('python src/main.py --config=LANL_inductive_exp0')` |
| Figures 9 and 10, runtime and memory | requires the baselines' own repositories |
| Figure 12, ACSAC'25 enterprise dataset | third-party dataset, not redistributed here |

Baselines (EULER, ARGUS, JBEIL) are not rerun; their numbers come from the paper and the
repositories cited in §IV-B.

## Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `gdown` fails or returns nothing | The file is not shared as *Anyone with the link*, or hit Google's 24h download quota. |
| Compilation was interrupted | Delete the half-written folders, `!rm -rf $LARES_DATA_ROOT/*/compiled/*/{LANL,OPTC}_exp*`, and run all cells again. |
| Session disconnected | Run all cells again: the download and finished compilations are detected and skipped. |
| `CUDA out of memory` | Switch to a CPU runtime. |